# 03 - Run the pipeline on the videos (Kaggle GPU) and collect the deliverables
**Add inputs:** (1) `roadscan-code`, (2) your **private** videos dataset, (3) the weights - either the output of notebook 01
or a dataset containing `yolo26s_rdd.pt`.  **Internet = On** (SegFormer / SAM 2 weights download).
Keep the videos dataset private and delete it from Kaggle after you have submitted.

In [ ]:
# --- locate the code dataset, copy it to /kaggle/working and install pinned deps ---
import glob, io, os, shutil, subprocess, sys
REPO = "/kaggle/working/road-defect-analyzer"
if not os.path.exists(REPO):
    # only the real dataset (never a copy of the code inside another notebook's output)
    hits = (glob.glob("/kaggle/input/datasets/**/roadscan/__init__.py", recursive=True)
            or glob.glob("/kaggle/input/**/roadscan/__init__.py", recursive=True))
    assert hits, "Add the 'roadscan-code' dataset via 'Add Input' (see docs/KAGGLE_RUNBOOK.md)"
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), REPO)
os.chdir(REPO)
!pip install -q ultralytics==8.4.177 lapx "transformers>=4.44" matplotlib
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import shlex
assert os.path.exists(REPO + "/roadscan/furniture.py"), "an OLD version of the roadscan-code dataset is attached - re-add it"
# Large files were split into byte-exact chunks (name.mp4.part0, part1, ...) to pass the upload limit.
# Rebuild them into normal .mp4 files; the result is byte-identical to the originals.
os.makedirs("/kaggle/working/videos", exist_ok=True)
groups = {}
for part in glob.glob("/kaggle/input/**/*.mp4.part*", recursive=True):
    groups.setdefault(part.rsplit(".part", 1)[0], []).append(part)
for base, parts in groups.items():
    with open("/kaggle/working/videos/" + os.path.basename(base), "wb") as out:
        for part in sorted(parts, key=lambda q: int(q.rsplit(".part", 1)[1])):
            with open(part, "rb") as src:
                shutil.copyfileobj(src, out)
    print("rebuilt", os.path.basename(base), "from", len(parts), "chunks")
videos = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True)
                + glob.glob("/kaggle/working/videos/*.mp4"))
# prefer the continued-training weights (notebook 04) when present
WEIGHTS = ((glob.glob("/kaggle/input/**/yolo26s_rdd_v2.pt", recursive=True)
            or glob.glob("/kaggle/input/**/yolo26s_rdd.pt", recursive=True)) or [None])[0]
print("videos :", *videos, sep="\n  ")
print("weights:", WEIGHTS)
assert videos and WEIGHTS
V = " ".join(shlex.quote(v) for v in videos)

In [ ]:
# --- main run: defects + road surface (paved/unpaved/sidewalk) + water-filled potholes + manholes/drains
#     + lane markings / crosswalks / signs / traffic lights (condition) + vehicles + route map.
#     All temporal smoothing is causal (past frames only). Higher input size and a lower confidence floor favour recall (the tracker + report
#     filter then decide what survives). Camera height is only used for the schematic map (1.4 m dashcam, 1.0 m low camera).
for v in videos:
    cam = 1.0 if "(1)" in os.path.basename(v) else 1.4
    !python scripts/run_video.py {shlex.quote(v)} --weights {WEIGHTS} --sam --scene --openvocab --furniture \
        --set detect.imgsz=800 --set detect.conf=0.15 --cam-height {cam} --out /kaggle/working/outputs

In [ ]:
import json, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
for d in sorted(glob.glob("/kaggle/working/outputs/*/")):
    info = json.load(open(d + "run_info.json"))
    print("\n==", info["video"])
    print("FPS (detect+road+track):", info["timing"]["processing_fps"], "| all models:", info["timing"].get("processing_fps_all_models"),
          "| end-to-end pass 1:", info["timing"]["end_to_end_fps_pass1"])
    print("road surface:", json.dumps(info.get("road_surface", {}).get("findings") if info.get("road_surface") else None))
    print("odometry:", info.get("odometry"))
    print("temporal:", info.get("temporal"))
    print("furniture:", json.dumps((info.get("furniture") or {}).get("marking", {}).get("mean_visibility")), "inventory items:",
          (info.get("furniture") or {}).get("inventory_items"))
    print("hardware:", info["hardware"]["gpus"], "|", info["hardware"]["cpu"])
    print("tracks:", info["tracking"], "| road filter:", info["road_filter"])
    print(pd.read_csv(d + "defects.csv")[["track_id","defect_type","first_seen_timecode","max_confidence","water_filled","surface_at_defect","map_x_m","map_y_m"]].head(60))

### Optional: ablations (counts + speed for each variant) and a TensorRT benchmark

In [ ]:
RUN_SWEEP = False      # ablations were already produced in an earlier run; set True to repeat them (~20+ min)
if RUN_SWEEP:
    !python scripts/sweep.py {V} --weights {WEIGHTS} --out /kaggle/working/outputs_sweep

In [ ]:
shutil.copy(WEIGHTS, "/kaggle/working/bench_weights.pt")      # export writes next to the weights: /kaggle/input is read-only
!python scripts/benchmark.py {shlex.quote(videos[0])} --weights /kaggle/working/bench_weights.pt --formats pt onnx engine --out /kaggle/working/benchmark_t4.json

### Optional: RF-DETR challenger (needs `rfdetr_small_rdd.pth` from notebook 02)

In [ ]:
RF = (glob.glob("/kaggle/input/**/rfdetr_small_rdd.pth", recursive=True) or [None])[0]
if RF:
    !pip install -q 'rfdetr[train,loggers]>=1.11'
    !python scripts/run_video.py {V} --backend rfdetr --weights {RF} --out /kaggle/working/outputs_rfdetr

In [ ]:
shutil.make_archive("/kaggle/working/submission_outputs", "zip", "/kaggle/working/outputs")
print(os.path.getsize("/kaggle/working/submission_outputs.zip") / 1e6, "MB")